In [ ]:
# 导包
import torch
import torch.nn as nn
import jieba
import pandas as pd
import numpy as np
import torch.optim as optim
import re
import torch.nn.functional as F
import time
from torch.utils.data import DataLoader,Dataset
from torchsummary import summary


In [3]:
# 导入文件并对其进行切割得到词向量
def build_vocab():
    # 数据集导入
    with open("jaychou_lyrics.txt",'r',encoding='utf8') as f1:
        # 构建词表
        all_words = []    
        unique_words = []
        for line in f1:
            words = jieba.lcut(line)
            all_words.append(words)
            # 去重
            for word in words:
                if word not in unique_words:
                    unique_words.append(word)    
        word_count = len(unique_words)
    # 词到索引的映射 字典推导式
    word_to_index = {word:idx for idx,word in enumerate(unique_words)} 
    # 词列索引表示  
    corpus_idx = []
    # 遍历每一行的分词结果
    for words in all_words:
        temp = []
        for word in words:
            temp.append(word_to_index[word])
        # 获取当前文档的词索引
        temp.append(word_to_index[' '])
        # 因为是将列表添加 所以试用的extent
        corpus_idx.extend(temp)
    return unique_words,word_count,word_to_index,corpus_idx    


In [4]:
# 构建数据集 继承Dataset
class LyricsDataset(Dataset):
    # num_chars：每组取几个数字
    def __init__(self, corpus_idx, num_chars):
        # 索引表达的语料 句子的索引
        self.corpus_idx = corpus_idx
        # 句子的长度
        self.num_chars = num_chars
        # 总共有多少个词语 不去重
        self.total_words = len(corpus_idx)
        # 总共有多少个句子 滑动切（重叠）和固定的切（不重叠）
        # 固定切
        self.total_samples = self.total_words // self.num_chars
        # 滑动窗口
        # self.total_samples = self.total_words - self.num_chars
    def __len__(self):
        # 返回样本数量
        return self.total_samples
    def __getitem__(self, idx):
        # 定义开始的节点
        start = min(max(idx,0), self.total_samples - 1)
        # 滑动窗口
        # start = min(max(idx,0), self.total_samples - self.num_chars - 1)
        # 定义输入
        x = self.corpus_idx[start*self.num_chars:(start+1)*self.num_chars]
        # x = self.corpus_idx[start:start+self.num_chars]
        # 定义输出
        y = self.corpus_idx[start*self.num_chars+1:(start+1)*self.num_chars+1]
        # y = self.corpus_idx[start+1:start+self.num_chars+1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long) 

In [5]:
# 构建神经网络
class RnnModel(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super(RnnModel,self).__init__()
        self.hidden_size = hidden_size
        # 创建词嵌入层
        # 参数：嵌入词的数量 嵌入词的维度 
        self.embe = nn.Embedding(output_size,input_size)
        # RNN层
        # 参数：嵌入词的维度 隐藏层的维度 隐藏层层数
        self.rnn = nn.RNN(input_size,hidden_size)
        # 创建输出层 多分类的任务
        self.out = nn.Linear(hidden_size,output_size)
    def forward(self,x,hidden):
        # 词嵌入
        # 词嵌入的形状(batch_size,seq_len,input_size)
        embedding = self.embe(x)
        # 循环网络计算
        # 参数：输入张量 词嵌入的结果 输入的形状(seq_len,batch_size,input_size)
        output,hidden = self.rnn(torch.transpose(embedding,0,1),hidden)
        # 输出层计算 将seq_len和batch_size合并了
        out = self.out(output.reshape((-1,output.shape[-1])))
        return out,hidden
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,self.hidden_size)

In [6]:
# 模型训练
def train():
    # 加载数据
    unique_words,word_count,word_to_index,corpus_idx = build_vocab()
    # 创建数据集
    dataset = LyricsDataset(corpus_idx,32)
    # 数据加载 数据集和数据加载的bt必须要成倍数的关系
    datalodader = DataLoader(dataset,128)
    # 创建模型对象
    model = RnnModel(128,128,word_count)
    # 创建损失函数
    loss_fn = nn.CrossEntropyLoss()
    # 创建优化器
    optimizer = optim.Adam(model.parameters(),0.001)
    # 开始训练
    epochs = 10
    for epoch in range(epochs):
        total_loss = 0
        start = time.time()
        iterater = 0
        for x,y in datalodader:
            # 向前传递 试用y.size(0)避免了最后一批的大小不足batch_size
            h0 = model.init_hidden(y.size(0))
            y_pred,h1 = model(x, h0)
            # 梯度清零
            optimizer.zero_grad()
            # 反向传播
            y = torch.transpose(y,0,1).contiguous().view(-1)
            loss = loss_fn(y_pred,y)
            loss.backward()
            total_loss += loss
            # 梯度更新
            optimizer.step()
        print(f"epoch:{epoch+1}loss:{total_loss/len(y)}time:{time.time()-start}")
    print("训练完成")
    # 保存模型
    torch.save(model.state_dict(),"geciyuce.pth")

In [7]:
# 模型测评
def model_eva(start_word, sentence_len):
    # 加载数据
    unique_words,word_count,word_to_index,corpus_idx = build_vocab()
    # 创建模型
    model = RnnModel(128,128,word_count)
    # 模型加载
    model.load_state_dict(torch.load("geciyuce.pth"))
    # 初始化隐藏状态
    hidden = model.init_hidden(1)
    # 输入文本转换为索引
    word_idx = word_to_index[start_word]
    # 生成的句子
    sentence = [word_idx]
    # 开始测评
    for _ in range(sentence_len):
        y_pred, hidden = model(torch.tensor([[word_idx]]),hidden)
        word_idx = torch.argmax(y_pred,1).item()
        sentence.append(word_idx)
    # 打印所有的歌词
    for word in sentence:
        print(unique_words[word],end='')

In [ ]:
# 测试
if __name__=='__main__':
    unique_words,word_count,word_to_index,corpus_idx = build_vocab()
    # 去重后的词标
    print(unique_words[:10])
    # 相当于一个新华字典 告诉rnn哪个词对应的索引是多少
    print(word_to_index)
    print(word_count) 
    # 当前文档中每个词对应的索引
    print(corpus_idx[:10])
    print(len(corpus_idx))
    # 构建数据集
    dataset = LyricsDataset(corpus_idx,5)
    x,y = dataset.__getitem__(0)
    print(x)
    print(y)
    # 创建模型
    model = RnnModel(input_size=128,hidden_size=128,output_size=word_count)
    h0 = model.init_hidden(1)
    y,h1 = model(x.unsqueeze(0), h0)
    print(y.shape)
    # 训练模型
    # train()
    # 模型预测
    model_eva("宇宙",10)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\hjh\AppData\Local\Temp\jieba.cache
Loading model cost 0.474 seconds.
Prefix dict has been built successfully.


['想要', '有', '直升机', '\n', '和', '你', '飞到', '宇宙', '去', '融化']
{'想要': 0, '有': 1, '直升机': 2, '\n': 3, '和': 4, '你': 5, '飞到': 6, '宇宙': 7, '去': 8, '融化': 9, '在': 10, '一起': 11, '里': 12, '我': 13, '每天': 14, '想想': 15, '著': 16, '这样': 17, '的': 18, '甜蜜': 19, '让': 20, '开始': 21, '相信': 22, '命运': 23, '感谢': 24, '地心引力': 25, '碰到': 26, '漂亮': 27, '面红': 28, '可爱': 29, '女人': 30, '温柔': 31, '心疼': 32, '透明': 33, '感动': 34, '坏坏': 35, '疯狂': 36, '乡': 37, '如果说': 38, '怀疑': 39, ' ': 40, '可以': 41, '造句': 42, '分离': 43, '能够': 44, '翻译': 45, '如果': 46, '这': 47, '一切': 48, '真的': 49, '将': 50, '寂寞': 51, '封闭': 52, '然后': 53, '这里': 54, '不': 55, '限': 56, '日期': 57, '过去': 58, '慢慢': 59, '温习': 60, '爱上你': 61, '那场': 62, '悲剧': 63, '是': 64, '完美': 65, '演出': 66, '一场': 67, '戏': 68, '宁愿': 69, '心碎': 70, '哭泣': 71, '再': 72, '狠狠': 73, '忘记': 74, '爱过': 75, '证据': 76, '晶莹': 77, '泪滴': 78, '闪烁': 79, '成': 80, '回忆': 81, '伤人': 82, '美丽': 83, '完美主义': 84, '太': 85, '彻底': 86, '连': 87, '恨': 88, '都': 89, '难以': 90, '下笔': 91, '真心': 92, '抽离': 93, '写成': 94, '日记': 95, '像是': 96